# Urbora YieldMax — train and test actual ML models

Run all cells on **CPU** in Kaggle or Google Colab. The notebook reads two SQLite databases from the pinned public Urbora repository, filters district observations, fits fresh scikit-learn estimators, and evaluates them on later crop years. The prediction cell calls **the fitted estimators** for the crop, district and horizon you enter.

- **Models:** Regularized ridge with a shrunk log-change estimate, and histogram gradient-boosted regression trees.
- **Target:** Future district yield in kg per hectare for lentil or the combined rapeseed/mustard category. Each horizon fits its own model.
- **Benchmarks:** Last observed yield and Urbora's damped Holt method. The notebook selects a model using earlier seasons and displays how both learned estimators and both benchmarks performed.
- **Access:** Internet for the initial public database download and package installation if needed. If internet is disabled, attach the two exact pinned SQLite files and set the data-directory path below. Colab and Kaggle both support CPU notebooks.
- **Scope:** This is a forecasting experiment and an inference interface for teammates. Its outputs represent district planning estimates. The historical benchmark was used in earlier Urbora experiments. Establish performance on fresh seasons before promoting a model to production.

[Urbora training inputs](https://github.com/Shafin2954/urbora/tree/68dfdcd731ddcd4f451ea801ea629056c377536d/data) · [scikit-learn boosting estimator](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import statistics
import subprocess
import sys
from urllib.request import urlopen

# Set this to the folder containing both database files when running offline.
# Kaggle datasets mounted under /kaggle/input are also discovered automatically.
DATA_DIRECTORY = os.environ.get("URBORA_DATA_DIR", "").strip()
ALLOW_DOWNLOAD = os.environ.get("YIELDMAX_ALLOW_DOWNLOAD", "1") == "1"
OUTPUT_DIRECTORY = Path(os.environ.get("YIELDMAX_OUTPUT_DIR", "yieldmax_training")).resolve()
SKLEARN_VERSION = "1.9.1"

try:
    installed = version("scikit-learn")
except PackageNotFoundError:
    installed = None
if installed != SKLEARN_VERSION:
    if not ALLOW_DOWNLOAD:
        raise RuntimeError(f"Offline setup needs scikit-learn=={SKLEARN_VERSION} already installed.")
    if "sklearn" in sys.modules:
        raise RuntimeError("Restart the runtime, then run this setup cell before importing scikit-learn.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--only-binary=:all:", "--timeout", "75",
        f"scikit-learn=={SKLEARN_VERSION}",
    ])

import joblib
import numpy as np
import scipy
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

assert sklearn.__version__ == SKLEARN_VERSION
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print(f"Python {sys.version.split()[0]} | scikit-learn {sklearn.__version__} | NumPy {np.__version__}")

## 1. Get and verify the training data

Only the pinned public data files are fetched. Kaggle users can attach these files as inputs and run without internet once the dependency is present.

In [ ]:
SOURCE_COMMIT = "68dfdcd731ddcd4f451ea801ea629056c377536d"
DATA_HASHES = {
    "agristats.db": "ba81185a45969d893e409c781ffb658810f2cf446bc5583cbd6b4e4fb43e8ff5",
    "agribase_clean.db": "168ffa3a7438566fde315eed29a84974cdff2fa7db38eb50ba278268ef156f25",
}
PINNED_FILES = f"https://raw.githubusercontent.com/Shafin2954/urbora/{SOURCE_COMMIT}/data"


def sha256(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()


def checked_data(directory):
    for name, expected in DATA_HASHES.items():
        path = directory / name
        if not path.is_file() or sha256(path) != expected:
            raise ValueError(f"{path} is missing or differs from the pinned Urbora snapshot.")
    return directory


def locate_data():
    if DATA_DIRECTORY:
        return checked_data(Path(DATA_DIRECTORY).expanduser().resolve())
    candidates = [Path.cwd() / "Source" / "data", Path.cwd() / "data"]
    mounted = Path("/kaggle/input")
    if mounted.is_dir():
        candidates.extend(path.parent for path in sorted(mounted.rglob("agristats.db")))
    for directory in candidates:
        if all((directory / filename).is_file() for filename in DATA_HASHES):
            try:
                return checked_data(directory)
            except ValueError:
                continue
    cache = OUTPUT_DIRECTORY / "source_cache" / "data"
    if all((cache / filename).is_file() for filename in DATA_HASHES):
        return checked_data(cache)
    if not ALLOW_DOWNLOAD:
        raise RuntimeError("Attach the pinned SQLite files, set DATA_DIRECTORY to their folder, and run again.")
    cache.mkdir(parents=True, exist_ok=True)
    for name, expected in DATA_HASHES.items():
        final = cache / name
        if final.is_file() and sha256(final) == expected:
            continue
        temp = final.with_suffix(".download")
        try:
            print(f"Downloading and checking {name} from the pinned commit...")
            with urlopen(f"{PINNED_FILES}/{name}", timeout=90) as response, temp.open("wb") as file:
                shutil.copyfileobj(response, file)
            if sha256(temp) != expected:
                raise ValueError(f"Source checksum failed for {name}.")
            temp.replace(final)
        except OSError as exc:
            raise RuntimeError("Could not download the source data. Enable internet or set DATA_DIRECTORY.") from exc
        finally:
            temp.unlink(missing_ok=True)
    return checked_data(cache)


data_directory = locate_data()
print("Both source checksums verified.")

## 2. Filter records and create lag features

Use only complete district rows that pass the original extract flags, crop bounds, and area/production/yield checks. Each forecast's features come from editions available by its cutoff.

In [ ]:
"""Read and vet district crop records from Urbora's agricultural yearbook extract."""

from __future__ import annotations

from collections import Counter, defaultdict
from contextlib import closing
from dataclasses import dataclass
import math
from pathlib import Path
import sqlite3
from statistics import median

CROPS = {
    "lentil_masur": (243, "Lentil (Masur)", 500, 5000),
    "rapeseed_mustard": (248, "Rape and Mustard (Local+HYV)", 500, 8000),
}
FIRST_YEAR = 2018
LAST_EDITION = 2025


@dataclass(frozen=True)
class Observation:
    crop: str
    district: str
    year: int
    edition: int
    value: float
    row_id: int


def connect(db: Path) -> sqlite3.Connection:
    return sqlite3.connect(db.resolve(strict=True).as_uri() + "?mode=ro", uri=True)


def load_observations(repo: Path) -> tuple[list[Observation], dict]:
    with closing(connect(repo / "data" / "agribase_clean.db")) as connection:
        districts = {name.casefold(): name for name, in connection.execute(
            "SELECT canonical_district FROM district_master"
        )}

    spec_by_id = {spec[0]: (crop, spec) for crop, spec in CROPS.items()}
    counts: Counter[str] = Counter()
    candidates: dict[tuple, list[Observation]] = defaultdict(list)
    with closing(connect(repo / "data" / "agristats.db")) as connection:
        for row in connection.execute(
            """
            SELECT f.crop_production_id, c.crop_id, c.crop_name, r.region_name,
                   y.start_year, y.end_year, f.yearbook_year,
                   f.area_hectares, f.production_metric_tons, f.yield_kg_per_hectare,
                   f.area_hectares_is_outlier, f.production_metric_tons_is_outlier,
                   f.yield_kg_per_hectare_is_outlier
            FROM fact_crop_production f
            JOIN dim_crop c USING(crop_id)
            JOIN dim_region r USING(region_id)
            JOIN dim_year y USING(reference_year_id)
            WHERE r.region_level = 'district' AND c.crop_id IN (?, ?)
            ORDER BY f.crop_production_id
            """,
            tuple(spec_by_id),
        ):
            row_id, crop_id, label, region, start, end, edition, area, production, value, *flags = row
            counts["candidate_rows"] += 1
            crop, (_, expected_name, minimum, maximum) = spec_by_id[crop_id]
            if label != expected_name:
                raise ValueError(f"Unexpected crop label for {crop_id}: {label}")
            district = districts.get(region.strip().casefold())
            if district is None:
                counts["noncanonical_district"] += 1
                continue
            if start is None or end is None or end - start != 1 or end != int(end) or not (
                FIRST_YEAR <= end <= edition <= LAST_EDITION
            ):
                counts["out_of_scope_year"] += 1
                continue
            if any(flags):
                counts["flagged_extract_outlier"] += 1
                continue
            if value is None or area is None or production is None or not all(
                math.isfinite(x) for x in (value, area, production)
            ) or not (minimum <= value <= maximum and area >= 10 and production > 0):
                counts["missing_or_outside_plausibility_filter"] += 1
                continue
            if abs(value / (1000 * production / area) - 1) > 0.015:
                counts["inconsistent_yield_arithmetic"] += 1
                continue
            key = crop, district, int(end), edition
            candidates[key].append(Observation(*key, value, row_id))

    kept = []
    for rows in candidates.values():
        values = [row.value for row in rows]
        middle = median(values)
        if (max(values) - min(values)) / middle > 0.01:
            counts["conflicting_same_edition_rows"] += len(rows)
            continue
        counts["agreeing_same_edition_duplicates"] += len(rows) - 1
        kept.append(min(rows, key=lambda row: (abs(row.value - middle), row.row_id)))
    counts["accepted_edition_rows"] = len(kept)
    if not kept:
        raise ValueError("No eligible crop observations in the supplied repository.")
    summary = {
        "quality_counts": dict(counts),
        "acceptance": (
            "Exact canonical district names, consecutive crop-year label, edition no earlier than harvest, "
            "unflagged complete rows, minimum 10 hectares, crop bounds, and yield arithmetic within 1.5%."
        ),
        "duplicates": "Within an edition, keep the published value closest to the median if the spread is at most 1%.",
        "vintage": "Take the newest eligible observation edition no later than the cutoff for each crop/district/harvest.",
    }
    return sorted(kept, key=lambda row: (row.crop, row.district, row.year, row.edition)), summary


class Panel:
    def __init__(self, rows: list[Observation]):
        self.rows = rows
        self.snapshots: dict[int, dict[tuple[str, str, int], Observation]] = {}

    def at(self, cutoff: int) -> dict[tuple[str, str, int], Observation]:
        if cutoff not in self.snapshots:
            self.snapshots[cutoff] = {
                (row.crop, row.district, row.year): row for row in self.rows if row.edition <= cutoff
            }
        return self.snapshots[cutoff]

    def histories(self, origin: int) -> dict[tuple[str, str], list[Observation]]:
        result: dict[tuple[str, str], list[Observation]] = defaultdict(list)
        for (crop, district, year), row in self.at(origin).items():
            if year <= origin:
                result[crop, district].append(row)
        return {
            key: rows for key, rows in result.items()
            if len(rows) >= 2 and rows[-1].year == origin
        }


def harvest_label(year: int) -> str:
    return f"{year - 1}-{year % 100:02d}"

In [ ]:
# Shared features, scoring and interval definitions from the local training pipeline.
SPLITS = {1: ((2022, 2023), (2024, 2025)), 2: ((2023,), (2025,))}


FEATURES = (
    "crop_is_rapeseed_mustard", "last_log_yield_vs_crop_median",
    "last_log_yield_vs_local_median", "yearly_last_log_change",
    "local_history_median_yearly_log_change", "crop_median_yearly_log_change",
)


HA_PER_UNIT = {"ha": 1.0, "bigha": 0.1337803776, "decimal": 0.0040468564224}


BAND_TARGET = 0.8


@dataclass(frozen=True)
class Features:
    history: list[Observation]
    values: list[float]

    @property
    def last(self) -> Observation:
        return self.history[-1]


def yearly_change(older: Observation, newer: Observation) -> float:
    return math.log(newer.value / older.value) / (newer.year - older.year)


def make_features(panel: Panel, origin: int) -> dict[tuple[str, str], Features]:
    histories = panel.histories(origin)
    levels: dict[str, list[float]] = defaultdict(list)
    trends: dict[str, list[float]] = defaultdict(list)
    for (crop, _), rows in histories.items():
        levels[crop].append(rows[-1].value)
        if rows[-2].year == origin - 1:
            trends[crop].append(yearly_change(rows[-2], rows[-1]))
    typical_yield = {crop: median(values) for crop, values in levels.items()}
    typical_change = {crop: median(values) for crop, values in trends.items()}
    result = {}
    for (crop, district), rows in histories.items():
        recent = rows[-4:]
        last = rows[-1]
        changes = [yearly_change(a, b) for a, b in zip(recent, recent[1:])]
        result[crop, district] = Features(rows, [
            float(crop == "rapeseed_mustard"),
            math.log(last.value / typical_yield[crop]),
            math.log(last.value / median(row.value for row in recent)),
            changes[-1],
            median(changes),
            typical_change.get(crop, 0.0),
        ])
    return result


def baseline_values(features: Features, horizon: int, fit_holt) -> dict:
    rows = features.history
    contiguous = [rows[-1].value]
    for earlier, later in zip(reversed(rows[:-1]), reversed(rows[1:])):
        if later.year != earlier.year + 1:
            break
        contiguous.append(earlier.value)
    contiguous.reverse()
    for _ in range(horizon):
        contiguous.append(fit_holt(contiguous)["base"])
    return {
        "last_observation": features.last.value,
        "trailing_median": median(point.value for point in rows[-3:]),
        "urbora_holt": contiguous[-1],
    }


def score(rows: list[dict], method: str) -> dict | None:
    if not rows or any(method not in row["predictions"] for row in rows):
        return None
    crops = defaultdict(list)
    for row in rows:
        crops[row["crop"]].append(row)
    per_crop = {}
    for crop, values in sorted(crops.items()):
        y = np.asarray([row["actual_kg_ha"] for row in values])
        yhat = np.asarray([row["predictions"][method] for row in values])
        error = yhat - y
        per_crop[crop] = {
            "rows": len(values),
            "mae_kg_ha": float(np.mean(abs(error))),
            "rmse_kg_ha": float(np.sqrt(np.mean(error ** 2))),
            "mean_actual_kg_ha": float(np.mean(y)),
            "wape_pct": float(np.sum(abs(error)) / np.sum(y) * 100),
        }
    return {
        "rows": len(rows),
        "mae_kg_ha": float(np.mean([abs(row["actual_kg_ha"] - row["predictions"][method]) for row in rows])),
        "macro_wape_pct": float(np.mean([item["wape_pct"] for item in per_crop.values()])),
        "by_crop": per_crop,
    }


def calibration(rows: list[dict], method: str) -> dict:
    residuals = defaultdict(list)
    for row in rows:
        error = abs(math.log(row["actual_kg_ha"] / row["predictions"][method]))
        residuals[row["crop"]].append(error)
    pooled = [value for crop in residuals.values() for value in crop]
    if not pooled:
        raise ValueError("No out-of-time calibration residuals.")
    result = {}
    for crop in CROPS:
        values = residuals[crop]
        used = sorted(values if len(values) >= 12 else pooled)
        rank = min(len(used), math.ceil((len(used) + 1) * BAND_TARGET))
        result[crop] = {"abs_log_residual": used[rank - 1], "n": len(used), "pooled": len(values) < 12}
    return result


def band(point: float, radius: float) -> tuple[float, float]:
    return point * math.exp(-radius), point * math.exp(radius)

In [ ]:
# Benchmark implementation copied from the audited Urbora checkout.
_ALPHA, _BETA, _PHI = 0.5, 0.3, 0.9


_MIN_HOLT_POINTS = 4          # below this, fall back to last-observed


_TREND_REL_EPS = 0.01         # |slope/level| under this → "flat"


def _classify_trend(slope: float, level: float) -> str:
    if level == 0:
        return "flat"
    rel = slope / level
    if rel > _TREND_REL_EPS:
        return "up"
    if rel < -_TREND_REL_EPS:
        return "down"
    return "flat"


def fit_series(ys: list[float]) -> dict:
    """Fit a series (ordered oldest→newest) → base yield, band, trend, method.

    * n >= 4 : damped Holt, base = 1-step-ahead forecast, band = stdev of in-sample
               one-step residuals, trend from the final damped slope.
    * 2 <= n < 4 : last-observed base, band = sample stdev, trend from overall slope.
    * n == 1 : base = the one value, band = 10% default, trend flat.
    """
    n = len(ys)
    if n == 1:
        return {"base": ys[0], "band": round(0.10 * ys[0], 2),
                "trend": "flat", "method": "single_point", "n": 1}

    if n < _MIN_HOLT_POINTS:
        slope = (ys[-1] - ys[0]) / (n - 1)
        band = statistics.pstdev(ys) if n >= 2 else 0.10 * ys[-1]
        return {"base": round(ys[-1], 2), "band": round(band, 2),
                "trend": _classify_trend(slope, ys[-1]), "method": "last_observed", "n": n}

    # damped Holt
    level = ys[0]
    trend = ys[1] - ys[0]
    resid: list[float] = []
    for t in range(1, n):
        forecast = level + _PHI * trend
        resid.append(ys[t] - forecast)
        new_level = _ALPHA * ys[t] + (1 - _ALPHA) * (level + _PHI * trend)
        trend = _BETA * (new_level - level) + (1 - _BETA) * _PHI * trend
        level = new_level
    base = level + _PHI * trend
    band = statistics.pstdev(resid) if len(resid) >= 2 else 0.10 * level
    return {"base": round(base, 2), "band": round(band, 2),
            "trend": _classify_trend(trend, level), "method": "damped_holt", "n": n}

In [ ]:
# The source loader expects a repo-style parent directory with a data/ subfolder.
# Conventional Kaggle inputs may use another folder name, so stage their SQLite files locally.
if data_directory.name != "data":
    source_workspace = OUTPUT_DIRECTORY / "notebook_data"
    data_for_training = source_workspace / "data"
    data_for_training.mkdir(parents=True, exist_ok=True)
    for name in DATA_HASHES:
        target = data_for_training / name
        if not target.is_file() or sha256(target) != DATA_HASHES[name]:
            shutil.copyfile(data_directory / name, target)
    data_directory = checked_data(data_for_training)
observations, data_audit = load_observations(data_directory.parent)
snapshot = Panel(observations).at(LAST_EDITION)
data_audit["unique_district_crop_year_observations"] = len(snapshot)
data_audit["districts_in_snapshot"] = len({obs.district for obs in snapshot.values()})
print(json.dumps(data_audit, indent=2))

## 3. Fit and score the ML estimators

The response is the log change from the latest observed yield. Models are refit independently per horizon and fold. The validation-selected ML estimator is scored alongside simpler baselines. The latest vintage is then used for the exported fit.

In [ ]:
"""Fitted CPU estimators for the portable Urbora YieldMax notebook."""





ML_METHODS = ("ridge_shrunk", "hist_gradient_boosting")
BENCHMARKS = ("last_observation", "urbora_holt")
PREDICTION_MULTIPLIERS = {"ridge_shrunk": 0.5, "hist_gradient_boosting": 1.0}


def fit_estimators(contexts: dict, panel: Panel, horizon: int, cutoff: int) -> tuple[dict, int]:
    observations = panel.at(cutoff)
    rows = [
        (features, observations[crop, district, target])
        for target in range(FIRST_YEAR + horizon, cutoff + 1)
        for (crop, district), features in contexts[target - horizon].items()
        if (crop, district, target) in observations
    ]
    if not rows:
        raise ValueError(f"No supervised training examples for horizon {horizon}, cutoff {cutoff}.")
    x = np.asarray([features.values for features, _ in rows], dtype=float)
    y = np.asarray([math.log(target.value / features.last.value) for features, target in rows])
    models = {
        "ridge_shrunk": make_pipeline(StandardScaler(), Ridge(alpha=30.0)),
        "hist_gradient_boosting": HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=100,
            max_leaf_nodes=7,
            max_depth=3,
            min_samples_leaf=16,
            l2_regularization=6.0,
            early_stopping=False,
            random_state=17,
        ),
    }
    with threadpool_limits(limits=2):
        for model in models.values():
            model.fit(x, y)
    return models, len(rows)


def predict_estimators(models: dict, features: list[Features]) -> dict:
    x = np.asarray([item.values for item in features], dtype=float)
    last = np.asarray([item.last.value for item in features], dtype=float)
    with threadpool_limits(limits=2):
        return {
            name: last * np.exp(PREDICTION_MULTIPLIERS[name] * model.predict(x))
            for name, model in models.items()
        }


def evaluate_estimators(panel: Panel, contexts: dict, fit_holt, horizon: int, years: tuple, truth_cutoff: int) -> list[dict]:
    truth = panel.at(truth_cutoff)
    evaluated = []
    for year in years:
        origin = year - horizon
        fold_models, train_n = fit_estimators(contexts, panel, horizon, origin)
        matches = [
            (features, truth[crop, district, year])
            for (crop, district), features in contexts[origin].items()
            if (crop, district, year) in truth
        ]
        if not matches:
            continue
        predicted = predict_estimators(fold_models, [item[0] for item in matches])
        for i, (features, target) in enumerate(matches):
            assert features.last.edition <= origin < target.edition <= truth_cutoff
            defaults = baseline_values(features, horizon, fit_holt)
            estimates = {name: defaults[name] for name in BENCHMARKS}
            estimates.update({name: float(values[i]) for name, values in predicted.items()})
            evaluated.append({
                "crop": target.crop,
                "district": target.district,
                "target_year": year,
                "origin_edition": origin,
                "horizon": horizon,
                "training_examples": train_n,
                "actual_kg_ha": target.value,
                "actual_row_id": target.row_id,
                "actual_edition": target.edition,
                "last_input_row_id": features.last.row_id,
                "predictions": estimates,
            })
    return evaluated


def train_cloud_models(panel: Panel, fit_holt) -> tuple[dict, dict, dict, list[dict]]:
    contexts = {year: make_features(panel, year) for year in range(FIRST_YEAR, LAST_EDITION + 1)}
    artifact = {"feature_names": list(FEATURES), "source_edition": LAST_EDITION, "horizons": {}}
    metrics = {"horizons": {}}
    predictions = []
    for horizon, (validation_years, test_years) in SPLITS.items():
        tuning_cutoff = min(test_years) - horizon
        assert max(validation_years) <= tuning_cutoff
        valid = evaluate_estimators(panel, contexts, fit_holt, horizon, validation_years, tuning_cutoff)
        test = evaluate_estimators(panel, contexts, fit_holt, horizon, test_years, LAST_EDITION)
        if not valid or not test:
            raise ValueError(f"Not enough evaluation coverage for horizon {horizon}.")
        validation = {name: score(valid, name) for name in BENCHMARKS + ML_METHODS}
        holdout = {name: score(test, name) for name in BENCHMARKS + ML_METHODS}
        selected = min(ML_METHODS, key=lambda name: validation[name]["macro_wape_pct"])
        intervals = {name: calibration(valid, name) for name in ML_METHODS}
        final_models, fit_count = fit_estimators(contexts, panel, horizon, LAST_EDITION)
        coverage = []
        for split, results in (("validation", valid), ("holdout", test)):
            for row in results:
                center = row["predictions"][selected]
                low, high = band(center, intervals[selected][row["crop"]]["abs_log_residual"])
                if split == "holdout":
                    coverage.append(low <= row["actual_kg_ha"] <= high)
                predictions.append(
                    {key: value for key, value in row.items() if key != "predictions"}
                    | row["predictions"]
                    | {"split": split, "selected_ml": selected, "low_kg_ha": low, "high_kg_ha": high}
                )
        artifact["horizons"][horizon] = {
            "trained_estimators": final_models,
            "estimators_train_rows": fit_count,
            "validation_selected_ml": selected,
            "validation_intervals": intervals,
        }
        metrics["horizons"][str(horizon)] = {
            "validation_years": validation_years,
            "validation_latest_eligible_edition": tuning_cutoff,
            "holdout_years": test_years,
            "selection": "Lowest validation macro-WAPE among the two fitted ML estimators.",
            "validation_selected_ml": selected,
            "validation": validation,
            "holdout": holdout,
            "selected_ml_band_coverage_pct": 100 * sum(coverage) / len(coverage),
            "final_training_examples": fit_count,
        }
    return artifact, metrics, contexts[LAST_EDITION], predictions


def supported_districts(features: dict, crop: str) -> list[str]:
    return sorted(district for label, district in features if label == crop)


def predict_crop_yield(
    artifact: dict, features: dict, *, crop: str, district: str, horizon: int, area: float,
    unit: str = "bigha", method: str | None = None,
) -> dict:
    if artifact["feature_names"] != list(FEATURES) or artifact["source_edition"] != LAST_EDITION:
        raise ValueError("Load the model from the matching notebook and pinned data edition.")
    if crop not in CROPS or unit not in HA_PER_UNIT or not math.isfinite(area) or area <= 0:
        raise ValueError("Enter a supported crop and unit and a positive finite area.")
    horizon_fit = artifact["horizons"].get(horizon)
    if horizon_fit is None:
        raise ValueError("Use a one- or two-year forecast horizon.")
    candidates = supported_districts(features, crop)
    district_name = next((name for name in candidates if name.casefold() == district.strip().casefold()), None)
    if district_name is None:
        raise ValueError(f"No eligible recent history for {crop} in {district}. Check supported_districts().")
    method = method or horizon_fit["validation_selected_ml"]
    if method not in horizon_fit["trained_estimators"]:
        raise ValueError(f"Choose one of the fitted estimators: {', '.join(ML_METHODS)}.")
    vector = features[crop, district_name]
    estimator = horizon_fit["trained_estimators"][method]
    point = float(predict_estimators({method: estimator}, [vector])[method][0])
    radius = horizon_fit["validation_intervals"][method][crop]["abs_log_residual"]
    low, high = band(point, radius)
    hectares = area * HA_PER_UNIT[unit]
    return {
        "crop": CROPS[crop][1],
        "district": district_name,
        "target_crop_year": harvest_label(LAST_EDITION + horizon),
        "origin_edition": LAST_EDITION,
        "model": method,
        "district_yield_kg_per_ha": round(point, 3),
        "district_yield_band_kg_per_ha": [round(low, 3), round(high, 3)],
        "area_ha": hectares,
        "illustrative_harvest_kg": round(point * hectares, 3),
        "illustrative_harvest_band_kg": [round(low * hectares, 3), round(high * hectares, 3)],
        "last_observation_kg_per_ha": vector.last.value,
        "last_observation_row_id": vector.last.row_id,
        "history_rows": len(vector.history),
    }

In [ ]:
fit_artifact, metrics, live_features, historical_predictions = train_cloud_models(Panel(observations), fit_series)
metrics["source"] = {
    "repository_commit": SOURCE_COMMIT,
    "data_sha256": DATA_HASHES,
    "table": "agristats.db::fact_crop_production",
    "edition": LAST_EDITION,
    "published_row_review": "Individual rows await reconciliation against the original yearbook PDFs.",
}
metrics["environment"] = {
    "python": sys.version.split()[0], "scikit_learn": sklearn.__version__,
    "numpy": np.__version__, "scipy": scipy.__version__, "joblib": joblib.__version__,
}
metrics["audit"] = data_audit

for horizon, item in metrics["horizons"].items():
    holdout = item["holdout"]
    chosen = item["validation_selected_ml"]
    lowest = min(holdout, key=lambda method: holdout[method]["mae_kg_ha"])
    print(f"\n{horizon}-year horizon | {item['final_training_examples']} final training examples")
    print(f"Validation-selected ML: {chosen} | Lowest holdout MAE: {lowest}")
    print(f"{'Estimator':28} {'MAE (kg/ha)':>13} {'Macro WAPE':>12} {'Rows':>6}")
    for name, series_metrics in holdout.items():
        print(f"{name:28} {series_metrics['mae_kg_ha']:13.2f} {series_metrics['macro_wape_pct']:11.2f}% {series_metrics['rows']:6d}")

## 4. Run a fresh inference

Change the district, crop, horizon or estimator below. The forecast reruns inference on that feature vector. Area changes the harvest calculation while the model predicts yield per hectare. Bands come from earlier forecast errors.

In [ ]:
# Edit these inputs and rerun this cell. The call uses the models just fitted above.
DISTRICT = "Rangpur"
CROP = "lentil_masur"  # Or "rapeseed_mustard"
HORIZON = 2          # 1 => crop year 2025-26, 2 => 2026-27, with the 2025 data cutoff
AREA = 2.0
UNIT = "bigha"       # Or "ha", "decimal"
METHOD = None        # None picks the validation-selected ML model. Or choose an ML_METHODS entry.

prediction_inputs = dict(crop=CROP, district=DISTRICT, horizon=HORIZON, area=AREA, unit=UNIT, method=METHOD)
prediction = predict_crop_yield(fit_artifact, live_features, **prediction_inputs)
print(json.dumps(prediction, indent=2, allow_nan=False))
print("Eligible districts for this crop:", ", ".join(supported_districts(live_features, CROP)))

## 5. Export and verify a trained artifact

The output folder contains fitted estimators and inference feature vectors, historical validation and holdout predictions, and metrics with input hashes and runtime versions.

In [ ]:
# Export weights and runtime feature vectors for a second session.
# Raw historical rows and any table of future estimates are outside this trained artifact.
input_vectors = {
    (crop, district): {"features": item.values, "latest_observed_yield": item.last.value,
                      "last_row_id": item.last.row_id, "history_rows": len(item.history)}
    for (crop, district), item in live_features.items()
}
export_payload = fit_artifact | {"source": metrics["source"], "environment": metrics["environment"],
                                  "inference_vectors": input_vectors}
artifact_path = OUTPUT_DIRECTORY / "yieldmax_ml.joblib"
joblib.dump(export_payload, artifact_path, compress=3)
# Only load a joblib artifact you created or trust. Reload with the same library versions.
reopened = joblib.load(artifact_path)
assert predict_crop_yield(reopened, live_features, **prediction_inputs) == prediction
# Smoke test for serving: recompute from a stored feature vector without a database query.
vector = reopened["inference_vectors"][CROP, prediction["district"]]
estimator = reopened["horizons"][HORIZON]["trained_estimators"][prediction["model"]]
with threadpool_limits(limits=2):
    fitted_delta = float(estimator.predict([vector["features"]])[0])
recomputed = vector["latest_observed_yield"] * math.exp(
    PREDICTION_MULTIPLIERS[prediction["model"]] * fitted_delta
)
assert round(recomputed, 3) == prediction["district_yield_kg_per_ha"]
(OUTPUT_DIRECTORY / "metrics.json").write_text(
    json.dumps(metrics, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
with (OUTPUT_DIRECTORY / "historical_predictions.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(historical_predictions[0]))
    writer.writeheader()
    writer.writerows(historical_predictions)
print("Saved trained estimators, metrics and historical predictions. Reloaded-model inference matched.")